## 🏋️ Exercise: Customer review analyzer

Companies drown in free-text reviews. Let's turn one into structured data your code can sort and chart.

Given this review:
```
"I ordered the wireless earbuds last week. Sound quality is amazing and battery
lasts all day, but the case feels cheap and it arrived two days late. Probably
won't buy from this brand again."
```

1. Create a Pydantic model `ReviewAnalysis` with these fields:
   - `sentiment`: `str`  (should come back as "positive", "negative", or "neutral")
   - `rating`: `int`  (the model's guess, 1 to 5)
   - `pros`: `list[str]`
   - `cons`: `list[str]`
2. Send the review with `response_schema=ReviewAnalysis`.
3. Print the sentiment and rating, then loop over `pros` and `cons` and print each one.

**Bonus 1:** make a list of 3 different reviews, loop over them, and print a one-line summary for each (`sentiment` + `rating`). Now you've got the core of a review-dashboard. 📊

**Bonus 2:** add a field `recommended_action: str` and watch the model suggest what the company should do.

Hint: the structure is identical to the `Recipe` example — only the fields change. 😉

In [2]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

In [7]:
from pydantic import BaseModel
from google.genai import types

# This class IS the shape we want back. Note the types: str, int, list of str.
class ReviewAnalysis(BaseModel):
    sentiment: str
    rating: int
    pros: list[str]
    cons: list[str]
    recommended_action: str

# Customer Review
review = """
I ordered the wireless earbuds last week. Sound quality is amazing
and battery lasts all day, but the case feels cheap and it arrived
two days late. Probably won't buy from this brand again.
"""

response = client.models.generate_content(
    model = "gemini-3.8-flash",
    contents = f"Analyze this customer review:\n{review}",
    
    config = types.GenerateContentConfig(
        response_mime_type = "application/json",         #Reply in JSON
        response_schema = ReviewAnalysis,               #Exaclty in this format
    ),
)

# Extract the structured response
analysis = response.parsed

# Print
print("Sentiment:" , analysis.sentiment)
print("Rating:" , analysis.rating, "/ 5")

print("\nPros:")
for pro in analysis.pros:
    print(" -", pro)

print("\nCons:")
for con in analysis.cons:
    print(" -", con)

print("\nRecommended Action:")
print(analysis.recommended_action)

Sentiment: Mixed
Rating: 2 / 5

Pros:
 - Amazing sound quality
 - Long-lasting all-day battery

Cons:
 - Cheaply made case
 - Delayed delivery (two days late)

Recommended Action:
Improve case material quality, optimize shipping fulfillment partners, and reach out to the customer with an apology or partial refund to regain trust.
